# System Testing

## System Overview

The system generates AHP weights from stakeholder preference ratings and then uses those weights as the input for the TOPSIS ranking process. Stakeholders do not provide traditional AHP pairwise comparisons by answering questions such as, “How much more important is criterion X than criterion Y?” Instead, the system asks stakeholders to rate each criterion independently, using linguistic values such as Very Low, Low, Medium, High, and Very High. These linguistic ratings are converted into numeric values, and the pairwise comparison matrix is then derived by comparing each criterion rating against every other criterion rating. For example, if a stakeholder rates Expected Crime Reduction as 5 and Implementation Cost as 3, the generated pairwise value is calculated as `5 / 3`. This creates a reciprocal pairwise matrix that can be processed by AHP, but it is important to note that the matrix is indirectly generated from independent criterion ratings rather than directly collected pairwise judgments.

After individual stakeholder matrices are generated, the system aggregates preferences by stakeholder type and applies stakeholder voting power when producing the final group pairwise matrix. AHP is then used to calculate the final criteria weights. These weights are passed into TOPSIS, where the scenario decision matrix is normalized, multiplied by the AHP-derived weights, and compared against the positive ideal and negative ideal solutions. The system calculates each alternative’s distance from the ideal and negative ideal solutions, computes the closeness coefficient, and ranks alternatives based on those coefficients.

## System Limitations

This system's process has several limitations that may affect results. Because the pairwise matrix is derived from independent ratings instead of direct criterion-to-criterion comparisons, the AHP weights may not fully represent the stakeholder’s actual trade-off reasoning. The linguistic scale also assumes that values such as Low, Medium, and High can be treated as evenly spaced numeric values, which may oversimplify stakeholder intent. In addition, the generated pairwise matrix may appear mathematically consistent because it is created from rating ratios, even though the stakeholder was not asked to make explicit pairwise judgments. Therefore, manual validation should compare results against the system’s derived-rating method, not against a traditional AHP survey process. These limitations should be considered when interpreting stability tests, rank reversals, and differences between system-generated, manually calculated, and coded validation results.

## Testing Overview

Now that the AHP-to-TOPSIS pipeline has been completed validation testing is required to confirm that the system produces accurate results. With the dashboard in place stakeholder preferences can be captured and processed as intended. This testing will compare system-generated results with manually calculated results to verify that the pipeline functions correctly and produces consistent outputs.

## Testing Scenario

The testing scenario and its data were created manually and are not based on any known policy scenarios.

### Scenario Description

A generic city is considering how to allocate a constrained public safety budget. Four distinct alternatives represent different strategic approaches: building new infrastructure, expanding existing capacity, increasing visibility through patrols, or investing in community-based prevention. This scenario brings together diverse stakeholders—from police leadership and city government to community advocates and residents—to weigh the trade-offs between cost, crime reduction effectiveness, community trust impact, and implementation speed. The decision involves balancing public safety improvements against resource constraints and community preferences for different intervention strategies.

### Alternatives

- Build new precinct
- Expand existing precinct
- Increase patrol units
- Invest in community violence prevention

### Criteria

- Implementation Cost
- Expected Crime Reduction
- Community Trust Impact
- Implementation Time

### Stakeholders

- Community Residents (0.15)
- Police Leadership (0.20)
- City Government (0.25)
- Community Advocates & Social Services (0.25)
- Business & Economic Development (0.15)

### Data

| alternative | cost | crime_reduction | community_trust | months_to_implement |
| ----------- | ---- | --------------- | --------------- | ------------------- |
| new_precinct | 2500000 | 10.7 | 89 | 30 |
| expand_precinct | 750000 | 3.8 | 91 | 21 |
| more_patrols | 250000 | 2.5 | 71 | 6 |
| community_prevention | 220000 | 2.6 | 68 | 7 |

---

In [1]:
import pandas as pd
import numpy as np

from pyDecision.algorithm import topsis_method, ahp_method
from IPython.display import display

def load_data(file_path):
    try:
        return pd.read_csv(file_path)
    except Exception as e:
        print(f"Error loading data: {e}")
        return None

def _convert_preference(pref):
    if pref == "Very Low":
        return 1
    elif pref == "Low":
        return 2
    elif pref == "Medium":
        return 3
    elif pref == "High":
        return 4
    elif pref == "Very High":
        return 5
    else:
        return None

def convert_linguistic_to_numeric(df):
    df["implementation_cost"] = df["implementation_cost"].apply(_convert_preference)
    df["expected_crime_reduction"] = df["expected_crime_reduction"].apply(_convert_preference)
    df["community_trust_impact"] = df["community_trust_impact"].apply(_convert_preference)
    df["implementation_time"] = df["implementation_time"].apply(_convert_preference)
    return df


def create_individual_pairwise_matrix(df):
    """
    Creates pairwise comparison matrices grouped by stakeholder_group.
    
    For each stakeholder group:
    - Builds individual 4x4 pairwise matrices for each stakeholder
    - Aggregates multiple stakeholders in the same group using geometric mean
    
    Returns a dict mapping stakeholder_group to aggregated 4x4 pairwise matrix.
    
    Args:
        df: DataFrame with columns [stakeholder_group, implementation_cost, expected_crime_reduction, 
                                   community_trust_impact, implementation_time]
    
    Returns:
        dict: {stakeholder_group: aggregated_pairwise_matrix_4x4}
    """
    criteria_cols = ["implementation_cost", "expected_crime_reduction", "community_trust_impact", "implementation_time"]
    
    # Group by stakeholder_group
    grouped = df.groupby("stakeholder_group")
    
    result = {}
    
    for stakeholder_group, group_df in grouped:
        # Extract numeric values for this stakeholder group
        group_data = group_df[criteria_cols].to_numpy()  # shape: (n_individuals, 4)
        
        # Build individual pairwise matrices for each stakeholder in the group
        individual_matrices = []
        
        for row in group_data:
            # Create 4x4 pairwise matrix: matrix[i][j] = row[i] / row[j]
            pairwise_matrix = np.zeros((4, 4))
            for i in range(4):
                for j in range(4):
                    pairwise_matrix[i][j] = float(row[i] / row[j])
            individual_matrices.append(pairwise_matrix)
        
        # Aggregate using geometric mean
        if len(individual_matrices) == 1:
            # Single stakeholder in group: use as-is
            aggregated_matrix = individual_matrices[0]
        else:
            # Multiple stakeholders: compute geometric mean element-wise
            # Geometric mean: (product) ^ (1/n)
            aggregated_matrix = np.ones((4, 4))
            for i in range(4):
                for j in range(4):
                    product = 1.0
                    for individual_matrix in individual_matrices:
                        product *= individual_matrix[i][j]
                    aggregated_matrix[i][j] = product ** (1.0 / len(individual_matrices))
        
        result[stakeholder_group] = aggregated_matrix
    
    return result


def create_aggregate_pairwise_matrix(grouped_matrices, stakeholder_weights):
    """
    Aggregates pairwise matrices across stakeholder groups using weighted geometric mean.
    
    Args:
        grouped_matrices: dict mapping stakeholder_group to 4x4 pairwise matrix
        stakeholder_weights: dict mapping stakeholder_group to voting_power (0-1)
    
    Returns:
        numpy array: 4x4 aggregated pairwise matrix
    """
    # Validate that weights sum to 1 (approximately)
    weights_sum = sum(stakeholder_weights.values())
    if weights_sum == 0:
        raise ValueError("Stakeholder weights sum to 0")
    
    # Renormalize weights to sum to 1
    normalized_weights = {k: v / weights_sum for k, v in stakeholder_weights.items()}
    
    # Initialize aggregated matrix with ones
    aggregated_matrix = np.ones((4, 4))
    
    # For each matrix position, compute weighted geometric mean
    for i in range(4):
        for j in range(4):
            product = 1.0
            for stakeholder_group, matrix in grouped_matrices.items():
                weight = normalized_weights.get(stakeholder_group, 0)
                if weight > 0:
                    # Use weight as exponent in geometric mean formula
                    product *= matrix[i][j] ** weight
            aggregated_matrix[i][j] = product
    
    return aggregated_matrix

if __name__ == "__main__":
    data = load_data("./system-testing-assets/data.csv")

    if data is None:
        raise ValueError("Failed to load required data file.")

    criteria = ["cost", "crime_reduction", "community_trust", "months_to_implement"]

    criteria_types = ["min", "max", "max", "min"]

    display(data)

,alternative,cost,crime_reduction,community_trust,months_to_implement
0,new_precinct,2500000,10.7,89,30
1,expand_precinct,750000,3.8,91,21
2,more_patrols,250000,2.5,71,6
3,community_prevention,220000,2.6,68,7


## Trials

The system will bes tested by conducting multiple trials and comparing the system-generated results with results obtained through manual calculations and coded validation. The tests will be divided into three groups, each with a different number of participants, and each group will then undergo stability testing.

### Rating Mapping

| Linguistic | Numeric |
| --------- | -------- |
| Very Low | 1
| Low | 2
| Medium | 3
| High | 4
| Very High | 5

### First Trial

The first trial will be a balanced pilot trial with one voice from each stakeholder group. This is best for manual verification because each stakeholder perspective is easy to trace.

#### Participants

| Participant | Stakeholder Group | Implementation Cost | Expected Crime Reduction | Community Trust Impact | Implementation Time | Story / Comment |
| --- | --- | --- | --- | --- | --- | --- |
| Jordan Reed | community_residents | High | Medium | Very High | Medium | Resident voice prioritizes community trust while still supporting practical safety improvements. |
| Maya Chen | business_economic | Medium | Very High | Low | Very High | Business owner wants rapid visible crime reduction to protect storefronts and customers. |
| Sofia Alvarez | police_leadership | Low | Very High | Medium | High | Public safety representative prioritizes crime reduction and operational speed over cost control. |
| Ethan Brooks | city_government | High | High | High | Medium | Planner prefers a balanced policy that manages cost, safety impact, and public legitimacy. |
| Ari Morgan | community_advocates | Medium | Low | Very High | Low | Civil rights advocate is cautious about rapid enforcement-heavy responses and emphasizes trust. |

#### Stakeholder Grouped Pairwise Matrices

In [2]:
trial1_test1_raw = {
    "participant": ["Jordan Reed", "Maya Chen","Sofia Alvarez", "Ethan Brooks", "Ari Morgan"],
    "stakeholder_group": ["community_residents", "business_economic", "police_leadership", "city_government", "community_advocates"],
    "implementation_cost": ["High", "Medium", "Low", "High", "Medium"],
    "expected_crime_reduction": ["Medium", "Very High", "Very High", "High", "Low"],
    "community_trust_impact": ["Very High", "Low", "Medium", "High", "Very High"],
    "implementation_time": ["Medium", "Very High", "High", "Medium", "Low"]
}

df = pd.DataFrame(trial1_test1_raw)
df = convert_linguistic_to_numeric(df)

# Create individual pairwise matrices grouped by stakeholder_group
grouped_matrices = create_individual_pairwise_matrix(df)

print("\n=== Grouped Pairwise Matrices by Stakeholder Group ===")
for stakeholder_group, matrix in grouped_matrices.items():
    print(f"\n{stakeholder_group}:")
    print(matrix)


=== Grouped Pairwise Matrices by Stakeholder Group ===

business_economic:
[[1.         0.6        1.5        0.6       ]
 [1.66666667 1.         2.5        1.        ]
 [0.66666667 0.4        1.         0.4       ]
 [1.66666667 1.         2.5        1.        ]]

city_government:
[[1.         1.         1.         1.33333333]
 [1.         1.         1.         1.33333333]
 [1.         1.         1.         1.33333333]
 [0.75       0.75       0.75       1.        ]]

community_advocates:
[[1.         1.5        0.6        1.5       ]
 [0.66666667 1.         0.4        1.        ]
 [1.66666667 2.5        1.         2.5       ]
 [0.66666667 1.         0.4        1.        ]]

community_residents:
[[1.         1.33333333 0.8        1.33333333]
 [0.75       1.         0.6        1.        ]
 [1.25       1.66666667 1.         1.66666667]
 [0.75       1.         0.6        1.        ]]

police_leadership:
[[1.         0.4        0.66666667 0.5       ]
 [2.5        1.         1.66666667 1.25

#### AHP Aggregate Weight Generation

In [3]:
# Define stakeholder voting power (from scenario.json)
stakeholder_weights = {
    "community_residents": 0.15, # Residents
    "business_economic": 0.15,  # Business & Economic Development
    "police_leadership": 0.20,  # Police Leadership
    "city_government": 0.25,  # City Government
    "community_advocates": 0.25  # Community Advocates & Social Services
}

# Create aggregate pairwise matrix weighted by stakeholder voting power
final_pairwise_matrix = create_aggregate_pairwise_matrix(grouped_matrices, stakeholder_weights)

print("\n=== Final Aggregated Pairwise Matrix ===")
print(final_pairwise_matrix)

# Compute AHP weights from aggregated pairwise matrix
ahp_weights, rc = ahp_method(final_pairwise_matrix, wd="mean")

print("\n=== AHP Weights ===")
for i in range(0, ahp_weights.shape[0]):
    print(f"{criteria[i]}: {ahp_weights[i]:.4f}")
    
print(f"\nConsistency Ratio (RC): {rc:.4f}")


=== Final Aggregated Pairwise Matrix ===
[[1.         0.89104228 0.83405895 1.00118653]
 [1.1222812  1.         0.93604868 1.12361282]
 [1.19895602 1.0683205  1.         1.20037862]
 [0.99881488 0.88998628 0.83307049 1.        ]]

=== AHP Weights ===
cost: 0.2315
crime_reduction: 0.2598
community_trust: 0.2775
months_to_implement: 0.2312

Consistency Ratio (RC): 0.0000


**Stakeholder Dashboard Results**

![Aggregated Pairwise Matrix](./system-testing-assets/trial1_test1_gpm.png)

![Aggregated Weighting Summary](./system-testing-assets/trial1_test1_gws.png)

#### AHP Results

The test results show that the outcomes are identical, confirming that the system correctly calculates weights for aggregated stakeholder groups.

#### TOPSIS Ranking

In [4]:
weights = [ahp_weights[0], ahp_weights[1], ahp_weights[2], ahp_weights[3]]

scores = topsis_method(data[criteria], weights, criteria_types, False, False)
data['Topsis Score'] = scores

rankings = data[['alternative', 'Topsis Score']].sort_values(by='Topsis Score', ascending=False)
print("\nTopsis Rankings:")
display(rankings)


Topsis Rankings:


,alternative,Topsis Score
2,more_patrols,0.575173
3,community_prevention,0.575027
1,expand_precinct,0.483431
0,new_precinct,0.423185


#### TOPSIS Results

### Second Trial

The second trial draws a more diverse mix of participants and presents a clearer narrative. Participation is moderate and business representation is stronger, but residents, planners, public safety officials, and civil rights advocates still offer important balance.

#### Participants

| Participant | Stakeholder Group | Implementation Cost | Expected Crime Reduction | Community Trust Impact | Implementation Time | Story / Comment |
| --- | --- | --- | --- | --- | --- | --- |
| Nia Thompson | community_residents | High | Medium | Very High | Medium | Resident favors trust-centered public safety with reasonable cost awareness. |
| Marcus Hill | community_residents | Medium | High | High | Medium | Resident wants crime reduction but not at the expense of community cooperation. |
| Priya Shah | community_residents | Very High | Low | High | High | Resident is cost-sensitive and prefers quick lower-cost interventions. |
| Caleb Johnson | community_residents | Medium | Medium | Very High | Low | Resident is willing to wait longer for a policy that improves legitimacy. |
| Denise Walker | business_economic | Medium | Very High | Low | Very High | Business owner strongly supports fast visible enforcement near commercial corridors. |
| Owen Patel | business_economic | Low | Very High | Medium | High | Business owner accepts higher spending if it lowers theft and disorder quickly. |
| Lena Torres | business_economic | Medium | High | Low | Very High | Business owner prioritizes speed and crime reduction over softer community metrics. |
| Victor Huang | business_economic | High | High | Medium | High | Business owner wants safety improvements but still notices budget limits. |
| Aisha Grant | business_economic | Medium | Very High | Medium | Very High | Business owner supports rapid response but wants enough trust to avoid backlash. |
| Samuel Price | business_economic | Low | Very High | Low | High | Business owner sees crime reduction as the dominant decision factor. |
| Grace Kim | police_leadership | Low | Very High | Medium | High | Official prioritizes operational effectiveness and rapid deployment. |
| Andre Coleman | police_leadership | Medium | Very High | High | High | Official wants strong crime reduction with enough community support for compliance. |
| Felix Romero | police_leadership | Low | High | Medium | Very High | Official values interventions that can be implemented quickly with measurable impact. |
| Mina Okafor | city_government | Very High | Medium | High | Medium | Planner emphasizes budget discipline and trust while avoiding overly reactive policy. |
| Nathan Lee | city_government | High | High | High | Medium | Planner supports a balanced capital and operational planning perspective. |
| Riley Carter | community_advocates | Medium | Low | Very High | Low | Advocate resists speed-driven enforcement and prioritizes public legitimacy. |
| Fatima Noor | community_advocates | High | Medium | Very High | Medium | Advocate accepts safety goals when paired with cost control and trust safeguards. |
| Elijah Stone | community_advocates | Medium | Low | Very High | Medium | Advocate prioritizes trust and careful implementation over aggressive enforcement. |

#### Stakeholder Grouped Pairwise Matrices

In [5]:
trial2_test1_raw = {
    "participant": ["Nia Thompson", "Marcus Hill", "Priya Shah", "Caleb Johnson", "Denise Walker", "Owen Patel", "Lena Torres", "Victor Huang", "Aisha Grant",
                    "Samuel Price", "Grace Kim", "Andre Coleman", "Felix Romero", "Mina Okafor", "Nathan Lee", "Riley Carter", "Fatima Noor", "Elijah Stone"],
    "stakeholder_group": ["community_residents", "community_residents", "community_residents", "community_residents", "business_economic", "business_economic",
                         "business_economic", "business_economic", "business_economic", "business_economic", "police_leadership", "police_leadership",
                         "police_leadership", "city_government", "city_government", "community_advocates", "community_advocates", "community_advocates"],
    "implementation_cost": ["High","Medium","Very High","Medium","Medium","Low","Medium","High","Medium","Low","Low","Medium","Low","Very High","High","Medium",
                            "High","Medium"],
    "expected_crime_reduction": ["Medium","High","Low","Medium","Very High","Very High","High","High","Very High","Very High","Very High","Very High","High",
                                 "Medium","High","Low","Medium","Low"],
    "community_trust_impact": ["Very High","High","High","Very High","Low","Medium","Low","Medium","Medium","Low","Medium","High","Medium","High","High",
                               "Very High","Very High","Very High"],
    "implementation_time": ["Medium","Medium","High","Low","Very High","High","Very High","High","Very High","High","High","High","Very High","Medium","Medium",
                            "Low","Medium","Medium"]
}

df = pd.DataFrame(trial2_test1_raw)
df = convert_linguistic_to_numeric(df)

# Create individual pairwise matrices grouped by stakeholder_group
grouped_matrices = create_individual_pairwise_matrix(df)

print("\n=== Grouped Pairwise Matrices by Stakeholder Group ===")
for stakeholder_group, matrix in grouped_matrices.items():
    print(f"\n{stakeholder_group}:")
    print(matrix)


=== Grouped Pairwise Matrices by Stakeholder Group ===

business_economic:
[[1.         0.59235304 1.12246205 0.61479778]
 [1.68818243 1.         1.89492071 1.03789082]
 [0.89089872 0.52772657 1.         0.54772256]
 [1.62655108 0.96349248 1.82574186 1.        ]]

city_government:
[[1.         1.29099445 1.11803399 1.49071198]
 [0.77459667 1.         0.8660254  1.15470054]
 [0.89442719 1.15470054 1.         1.33333333]
 [0.67082039 0.8660254  0.75       1.        ]]

community_advocates:
[[1.         1.44224957 0.66038545 1.25992105]
 [0.69336127 1.         0.4578857  0.87358046]
 [1.51426716 2.18395116 1.         1.90785707]
 [0.79370053 1.14471424 0.52414828 1.        ]]

community_residents:
[[1.         1.25743343 0.81903626 1.25743343]
 [0.79527073 1.         0.65135556 1.        ]
 [1.22094717 1.53525978 1.         1.53525978]
 [0.79527073 1.         0.65135556 1.        ]]

police_leadership:
[[1.         0.49324241 0.69336127 0.53132928]
 [2.02740067 1.         1.40572111 1.07

#### AHP Aggregate Weight Generation

In [6]:
# Define stakeholder voting power (from scenario.json)
stakeholder_weights = {
    "community_residents": 0.15, # Residents
    "business_economic": 0.15,  # Business & Economic Development
    "police_leadership": 0.20,  # Police Leadership
    "city_government": 0.25,  # City Government
    "community_advocates": 0.25  # Community Advocates & Social Services
}

# Create aggregate pairwise matrix weighted by stakeholder voting power
final_pairwise_matrix = create_aggregate_pairwise_matrix(grouped_matrices, stakeholder_weights)

print("\n=== Final Aggregated Pairwise Matrix ===")
print(final_pairwise_matrix)

# Compute AHP weights from aggregated pairwise matrix
ahp_weights, rc = ahp_method(final_pairwise_matrix, wd="mean")

print("\n=== AHP Weights ===")
for i in range(0, ahp_weights.shape[0]):
    print(f"{criteria[i]}: {ahp_weights[i]:.4f}")
    
print(f"\nConsistency Ratio (RC): {rc:.4f}")


=== Final Aggregated Pairwise Matrix ===
[[1.         0.97031585 0.85069966 0.99252079]
 [1.03059226 1.         0.87672449 1.02288424]
 [1.17550299 1.14060918 1.         1.16671116]
 [1.00753557 0.97762773 0.85711017 1.        ]]

=== AHP Weights ===
cost: 0.2373
crime_reduction: 0.2446
community_trust: 0.2790
months_to_implement: 0.2391

Consistency Ratio (RC): -0.0000


**Stakeholder Dashboard Results**

![Aggregated Results](./system-testing-assets/trial2_test1.png)

#### AHP Results

The test results show that the outcomes differ. My investigation found that the system calculates aggregate weights incorrectly by treating each submission as a separate voting unit. Each submission is converted into an AHP result, and its pairwise matrix and normalized voting power are added to the aggregation list. The system then renormalizes participant weights and calculates a single weighted geometric mean across all individual matrices.

**Incorrect Method:**

If all 18 participants are equal, each participant gets:

`1 / 18 = 0.0556`

So if community residents have 8 participants, their combined influence is:

`8 × 0.0556 = 0.4448`

That stakeholder type now controls about 44% of the aggregate result, instead of the 15% they are supposed to have.

**Correct Method:**

If there are 5 stakeholder groups with equal stakeholder-type weights, each group gets:

`1 / 5 = 0.20`

If community residents have 8 participants, then each resident participant gets:

`0.20 / 8 = 0.025`

If police leadership has 1 participant, that participant gets:

`0.20 / 1 = 0.20`

## Testing Outcome

Based on this finding, I need to make several changes to ensure the system reflects each group's intended voting power. I will then repeat the testing after those updates are complete.